# 1541. 平衡括号字符串的最少插入次数

题目：[力扣 1541](https://leetcode.cn/problems/minimum-insertions-to-balance-a-parentheses-string/)

一个 `(` 匹配两个连续的 `)`。和 1021 的普通括号匹配相比，关闭单位从 `)` 变成 `))`，输入也不再保证平衡。

## Solution 1：我的修正版，按右括号分组

`left_cnt` 是当前尚未匹配的左括号数，不是左括号总数。每组 `))` 消耗一个左括号；单独的 `)` 补齐一组，没有左括号就补一个 `(`。扫描完补 `2 * left_cnt`。

以下保留原代码逻辑。时间 O(n)，额外空间 O(1)。

In [1]:
class Solution:
    def minInsertions(self, s: str) -> int:
        left_cnt = 0
        insert_cnt = 0
        i = 0
        while i < len(s):
            if s[i] == "(":
                left_cnt+=1
                i+=1
            else:
                if left_cnt>0:
                    left_cnt-=1
                else:
                    insert_cnt+=1
                if i<len(s)-1:
                    if s[i+1] == ")":
                        i+=2
                    else:
                        i+=1
                        insert_cnt+=1
                else:
                    i+=1
                    insert_cnt+=1
        insert_cnt+=2*left_cnt
        return insert_cnt


## 两个实际踩坑

1. 检查 `s[i + 1]` 只需 `i < len(s) - 1`。原来的 `i < len(s) - 2` 会把 `())` 中现有的两个连续右括号拆开，错算成 3，正确是 0。
2. `(` 的答案是 2；不能直接返回 `insert_cnt`，要结算剩余左括号。1021 输入合法会自然归零，本题不会。

## Solution 2：逐字符统计欠下的右括号

`need` 表示当前还欠多少个右括号。遇到 `(`，若旧 `need` 是奇数，先补一个 `)` 完成旧半组，再 `need += 2`。遇到 `)` 先减一；若变成 -1，补一个 `(`，当前右括号已满足一半，令 `need = 1`。最后加上 `need`。

这份代码更简洁，两种解法的复杂度相同。单独提交时将 `Solution2` 改成 `Solution`。

In [2]:
class Solution2:
    def minInsertions(self, s: str) -> int:
        need = 0
        insert_cnt = 0

        for ch in s:
            if ch == "(":
                if need % 2 == 1:
                    insert_cnt += 1
                    need -= 1
                need += 2
            else:
                need -= 1
                if need == -1:
                    insert_cnt += 1
                    need = 1

        return insert_cnt + need


## 与 1021 对照：计数器为什么有效？

| 内容 | 1021 | 1541 |
| --- | --- | --- |
| 关闭单位 | 一个 `)` | 连续的 `))` |
| 计数含义 | 未闭合左括号数，即深度 | 未匹配左括号数，或右括号需求 |
| 处理右括号 | 一个字符令深度减一 | 一组字符令左计数减一 |
| 输入 | 保证合法 | 可能不平衡 |
| 末尾处理 | 深度自然归零 | 补 `2 * left_cnt` 或 `need` |
| 额外空间 | Python 结果列表 O(n) | 只返回次数 O(1) |

同类左括号栈只需要长度时，可以压缩成计数器。分组法一次完成整个关闭单位；逐字符法还要知道有没有半组，用 `need` 的奇偶性表达。多类型括号或需要匹配下标时，单一计数通常不够。

不能只算全串数量：`))(` 左 1 右 2，仍需插入 3 次；`倍数正确` 也不能替代连续性，`()()))` 的答案是 3。

详细题解与正确性说明：[1541 学习笔记](../note/leetcode_1541_minimum_insertions.md)。

## 样例与边界对照

比较 `ans == expected`，同时检查两种解法。

In [3]:
cases = [
    ("(()))", 1), ("())", 0), ("))())(", 3),
    ("((((((", 12), (")))))))", 5),
    ("(", 2), (")", 2), ("))", 1), ("()", 1),
    ("()(", 3), ("))(", 3), ("()()))", 3), (")()(", 5),
]
for solution in (Solution(), Solution2()):
    for s, expected in cases:
        ans = solution.minInsertions(s)
        assert ans == expected, (type(solution).__name__, s, ans, expected)
    print(f"{type(solution).__name__}: {len(cases)} 个样例全部通过")


Solution: 13 个样例全部通过
Solution2: 13 个样例全部通过
